In [1]:
import os
from google.colab import userdata

os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')


!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge

import zipfile
with zipfile.ZipFile("challenges-in-representation-learning-facial-expression-recognition-challenge.zip", "r") as zip_ref:
    zip_ref.extractall("fer2013_data")

print("Dataset downloaded and extracted successfully!")

100% 285M/285M [00:02<00:00, 140MB/s]

Dataset downloaded and extracted successfully!


In [2]:
import os

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [20]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class FERDataset(Dataset):
  def __init__(self, X, y):
      self.X = X.reset_index(drop=True)
      self.y = y.reset_index(drop=True)

  def __len__(self):
      return len(self.X)

  def __getitem__(self, idx):
      pixels = np.fromstring(self.X.iloc[idx], sep=' ', dtype=np.uint8)
      img = pixels.reshape(48, 48)
      img = torch.tensor(img, dtype=torch.float32).unsqueeze(0) / 255.0
      label = torch.tensor(self.y.iloc[idx], dtype=torch.long)
      return img, label

In [4]:
import pandas as pd

train_split = pd.read_csv('/content/drive/MyDrive/FER_project/data/train_split.csv')
val_split = pd.read_csv('/content/drive/MyDrive/FER_project/data/val_split.csv')



In [5]:
print(train_split.shape)
print(val_split.shape)


(22967, 2)
(5742, 2)


In [33]:
X_train = train_split["pixels"]
y_train = train_split["emotion"]

X_val = val_split["pixels"]
y_val = val_split["emotion"]



In [34]:
print(X_train.shape)
print(y_train.shape)
print(X_val.shape)
print(y_val.shape)

(22967,)
(22967,)
(5742,)
(5742,)


In [35]:
train_dataset = FERDataset(X_train, y_train)
val_dataset = FERDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [36]:
import torch
import torch.nn as nn


class BaseCNN(nn.Module):
  def __init__(self):
      super(BaseCNN, self).__init__()

      self.features = nn.Sequential(
          nn.Conv2d(1, 16, kernel_size=3, padding=1),
          nn.ReLU(),
          nn.MaxPool2d(2),

          nn.Conv2d(16, 32, kernel_size=3, padding=1),
          nn.ReLU(),
          nn.MaxPool2d(2),
      )

      self.classifier = nn.Sequential(
          nn.Flatten(),
          nn.Linear(32*12*12, 128),
          nn.ReLU(),
          nn.Linear(128,7)
      )

  def forward(self, x):
      x = self.features(x)
      x = self.classifier(x)
      return x



In [37]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BaseCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)


In [13]:
!pip install wandb -q

In [14]:
import wandb

wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nmetr23 (nmetr23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [15]:
wandb.init(
    project="FER-Challenge",
    name="base_model"
)

In [16]:
wandb.config.update({
    "lr": 1e-3,
    "batch_size":64,
    "model":"BaseCNN",
    "optimizer": "Adam",
    "epochs": 10
})

In [38]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)


In [39]:
def evaluate(model, loader, criterion, device):
    model.eval()
    correct = 0
    total = 0
    val_loss = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)

            outputs = model(x)
            loss = criterion(outputs, y)
            val_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            correct += (preds==y).sum().item()
            total += y.size(0)

    return val_loss / len(loader), correct / total



In [40]:
epochs = wandb.config.epochs

for epoch in range(epochs):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss":val_loss,
        "val_accuracy":val_acc
    })

    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}, val_acc={val_acc:.4f}")

Epoch 1: train_loss=1.6851, val_acc=0.3939
Epoch 2: train_loss=1.5338, val_acc=0.4298
Epoch 3: train_loss=1.4399, val_acc=0.4575
Epoch 4: train_loss=1.3628, val_acc=0.4690
Epoch 5: train_loss=1.2891, val_acc=0.4889
Epoch 6: train_loss=1.2174, val_acc=0.4913
Epoch 7: train_loss=1.1376, val_acc=0.4981
Epoch 8: train_loss=1.0596, val_acc=0.5068
Epoch 9: train_loss=0.9779, val_acc=0.5059
Epoch 10: train_loss=0.8957, val_acc=0.5094


In [41]:
wandb.finish()

epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▇▆▅▄▄▃▂▂▁
val_accuracy,▁▃▅▆▇▇▇███
val_loss,█▆▄▃▁▁▁▁▂▃
epoch,10
train_loss,0.89571
val_accuracy,0.5094
val_loss,1.38509


In [42]:
lrs = [1e-2, 1e-3, 1e-4]
batch_sizes = [32, 64]
optimizers = ["adam", "sgd"]

In [43]:
import itertools
import wandb

for lr, batch_size, opt_name in itertools.product(lrs, batch_sizes, optimizers):

    wandb.init(
        project="FER-Challenge",
        name=f"base_model (lr{lr}_bs{batch_size}_opt{opt_name})",
        reinit=True
    )

    wandb.config.update({
        "lr": lr,
        "batch_size": batch_size,
        "optimizer": opt_name
    })

    model = BaseCNN().to(device)

    if opt_name == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    for epoch in range(5):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        wandb.log({
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_accuracy": val_acc
        })

    wandb.finish()

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


train_loss,█▁▁▁▁
val_accuracy,▁▁▁▁▁
val_loss,▅█▁▆▂
train_loss,1.8118
val_accuracy,0.25131
val_loss,1.81099


train_loss,█▆▄▃▁
val_accuracy,▁▁▂▄█
val_loss,█▇▅▃▁
train_loss,1.74181
val_accuracy,0.30129
val_loss,1.72739


train_loss,█▁▁▁▁
val_accuracy,▁▁▁▁▁
val_loss,█▂▃▂▁
train_loss,1.81164
val_accuracy,0.25131
val_loss,1.81019


train_loss,█▆▄▂▁
val_accuracy,▁▂▃▅█
val_loss,█▆▄▂▁
train_loss,1.73244
val_accuracy,0.30425
val_loss,1.72643


...

train_loss,█▅▂▁▁
val_accuracy,▁▁▁▁▁
val_loss,█▄▂▁▁
train_loss,1.812
val_accuracy,0.25131
val_loss,1.80919


train_loss,█▅▃▂▁
val_accuracy,▁▅▇▇█
val_loss,█▅▃▂▁
train_loss,1.58873
val_accuracy,0.39829
val_loss,1.58058


train_loss,█▆▄▃▁
val_accuracy,▁█▇▇▇
val_loss,█▆▄▃▁
train_loss,1.91963
val_accuracy,0.25131
val_loss,1.91621


train_loss,█▅▃▂▁
val_accuracy,▁▄▇██
val_loss,█▅▃▂▁
train_loss,1.58117
val_accuracy,0.39429
val_loss,1.57796


train_loss,█▆▄▃▁
val_accuracy,▁▃███
val_loss,█▆▄▃▁
train_loss,1.90941
val_accuracy,0.25131
val_loss,1.90471
